In [2]:
!pip install transformers torch torchvision einops timm peft sentencepiece


In [3]:
# !pip install flash_attn
!pip uninstall flash_attn


In [4]:
import torch
from transformers import AutoModel, AutoTokenizer



In [5]:
model_name=  'h2oai/h2ovl-mississippi-2b'

model = AutoModel.from_pretrained(
    model_name,
    torch_dtype=torch.bfloat16,
    low_cpu_mem_usage=True,
    trust_remote_code=True).eval().cuda()
tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True, use_fast=False)
generation_config = dict(max_new_tokens=1024, do_sample=True)

/usr/local/lib/python3.10/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(
/usr/local/lib/python3.10/dist-packages/timm/models/layers/__init__.py:48: FutureWarning: Importing from timm.models.layers is deprecated, please import via timm.layers
  warnings.warn(f"Importing from {__name__} is deprecated, please import via timm.layers", FutureWarning)


FlashAttention is not installed.


In [6]:
device= torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

H2OVLChatModel(
  (vision_model): InternVisionModel(
    (embeddings): InternVisionEmbeddings(
      (patch_embedding): Conv2d(3, 1024, kernel_size=(14, 14), stride=(14, 14))
    )
    (encoder): InternVisionEncoder(
      (layers): ModuleList(
        (0-23): 24 x InternVisionEncoderLayer(
          (attn): InternAttention(
            (qkv): Linear(in_features=1024, out_features=3072, bias=True)
            (attn_drop): Dropout(p=0.0, inplace=False)
            (proj_drop): Dropout(p=0.0, inplace=False)
            (proj): Linear(in_features=1024, out_features=1024, bias=True)
          )
          (mlp): InternMLP(
            (act): GELUActivation()
            (fc1): Linear(in_features=1024, out_features=4096, bias=True)
            (fc2): Linear(in_features=4096, out_features=1024, bias=True)
          )
          (norm1): LayerNorm((1024,), eps=1e-06, elementwise_affine=True)
          (norm2): LayerNorm((1024,), eps=1e-06, elementwise_affine=True)
          (drop_path1): Identi

In [19]:

question="What is tenancy contract?"

response,history= model.chat(tokenizer, None, question, generation_config, history=None, return_history=True)
print(response)
print("-------")
print(history)

A tenancy contract, also known as a lease or rental agreement, is a legal document that outlines the terms and conditions of a rental property. It is a contract between a landlord and a tenant and is designed to protect both parties from potential disputes and misunderstandings.

The tenancy contract typically includes the following key provisions:

1. **Rental Terms**: The duration of the tenancy, the number of rooms, and the rent amount are specified.
2. **Lease Terms**: The notice period, quiet enjoyment rights, and other essential terms are outlined.
3. **Termination and Renewal**: The conditions under which the tenancy can be terminated and renewed.
4. **Property Maintenance**: The landlord's responsibilities for maintaining the property, including repairs and renovations.
5. **Security Deposit**: The required security deposit and how it is returned to the tenant at the end of the tenancy.
6. **Utility Connections**: The arrangement for securing utilities like electricity, water, 

In [21]:
from PIL import Image

image_file = '/content/___TMP.png'
# image = Image.open(image_file)


prompt = """
Extract all relevant information from the image in a structured, key-value format. Organize the data into the following main categories:
1. Owner Information
2. Tenant Information
3. Property Information
4. Contact Information
Ensure that the extracted data is comprehensive and clearly categorized for easy interpretation.
"""



response, history = model.chat(tokenizer, image_file, prompt, generation_config, history=None, return_history=True)
print(f'Assistant: {response}')



Assistant: {
    "Owner Information": {
        "Owner Name": "Ahmed Bin Saeed",
        "Emirates ID": "18419871234567",
        "Licensing Authority": "Department of Land",
        "Email": "mohammed.altayer@example.com",
        "Phone": "+971 50 123 4567"
    },
    "Tenant Information": {
        "Tenant Name": "Sarah Al Mansoori",
        "Emirates ID": "998765431990784",
        "Licensing Authority": "Department of Land",
        "Email": "sarah.almansoori@example.com",
        "Phone": "+971 9876543210"
    },
    "Property Information": {
        "Property Usage": [
            {"Type": "Residential", "Location": "Dubai Marina"},
            {"Type": "Apartment", "Location": "Dubai Marina"}
        ],
        "Property Type": "Apartment",
        "Building Name": "Marina Tower",
        "Property Number": "APT 1204",
        "Property Area (sqm)": "150 m²",
        "Premises No. (DEWA)": "9876543",
        "Property Area (sqm)": "150 m²",
        "Premises No. (DEWA)": "98765

In [9]:
!nvidia-smi


Wed Dec 11 14:42:10 2024       
+---------------------------------------------------------------------------------------+
| NVIDIA-SMI 535.104.05             Driver Version: 535.104.05   CUDA Version: 12.2     |
|-----------------------------------------+----------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id        Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |         Memory-Usage | GPU-Util  Compute M. |
|                                         |                      |               MIG M. |
|=========================================+======================+======================|
|   0  Tesla T4                       Off | 00000000:00:04.0 Off |                    0 |
| N/A   75C    P0              41W /  70W |   9621MiB / 15360MiB |     98%      Default |
|                                         |                      |                  N/A |
+-----------------------------------------+----------------------+--